# Code mẫu — Cây quyết định

Ta bắt đầu từ một phép chia nhỏ, đủ nhỏ để kiểm bằng tay. Sau đó mới chọn độ sâu và cắt tỉa cây trên một tập giữ lại. Mỗi lần cây phức tạp hơn hay đơn giản đi, ta đều đối chiếu trên dữ liệu chưa dùng để fit.

## Ví dụ 1: Tìm ngưỡng bằng Gini

Sáu hồ sơ được sắp theo một điểm số `x`, và nhãn 0 hoặc 1 cho biết hồ sơ thuộc nhóm nào. Ta tính Gini ở nút cha, thử từng ngưỡng nằm giữa hai giá trị liên tiếp, rồi chọn ngưỡng làm Gini giảm nhiều nhất.

In [ ]:
import numpy as np

x = np.array([1., 2., 3., 4., 5., 6.])
y = np.array([0, 0, 1, 0, 1, 1])

def gini(labels):
    counts = np.bincount(labels, minlength=2)
    p = counts / len(labels)
    return 1.0 - np.sum(p ** 2)

parent = gini(y)
thresholds = (x[:-1] + x[1:]) / 2
gains = []
for threshold in thresholds:
    left = x <= threshold
    right = ~left
    weighted_after = left.mean() * gini(y[left]) + right.mean() * gini(y[right])
    gain = parent - weighted_after
    gains.append((float(threshold), float(gain)))
    print("t =", threshold, "gain =", round(float(gain), 3),
          "sizes =", int(left.sum()), int(right.sum()))

best_threshold, best_gain = max(gains, key=lambda item: item[1])
print("Gini cha:", round(float(parent), 3))
print("ngưỡng tốt nhất:", best_threshold, "gain:", round(best_gain, 3))
assert best_threshold == 2.5
assert np.isclose(best_gain, 0.25)

t = 1.5 gain = 0.1 sizes = 1 5
t = 2.5 gain = 0.25 sizes = 2 4
t = 3.5 gain = 0.056 sizes = 3 3
t = 4.5 gain = 0.25 sizes = 4 2
t = 5.5 gain = 0.1 sizes = 5 1
Gini cha: 0.5
ngưỡng tốt nhất: 2.5 gain: 0.25


Gini của nút cha chỉ cần tính một lần. Gini sau khi chia là trung bình Gini của hai nhánh, có trọng số theo số mẫu. Nhờ trọng số này, một nhánh rất nhỏ dù hoàn toàn thuần cũng không tự động thắng. Ngưỡng 1.5 tách được một hồ sơ thuần nhóm 0, nhưng gain chỉ là 0.1.

Ngưỡng 2.5 và 4.5 cùng có gain 0.25. Hàm `max` giữ ngưỡng gặp trước trong danh sách, nên 2.5 được chọn. Đây chỉ là luật phá hòa, không có nghĩa ngưỡng 4.5 kém hơn.

Khi có giá trị trùng nhau, không được đặt ngưỡng giữa hai bản ghi bằng nhau. Ta chỉ giữ những cặp giá trị kế tiếp thật sự khác nhau, để khỏi thử lại một phép chia đã có.

In [ ]:
x_tied = np.array([1., 1., 2., 4.])
y_tied = np.array([0, 1, 1, 1])
all_midpoints = (x_tied[:-1] + x_tied[1:]) / 2
valid_midpoints = all_midpoints[x_tied[:-1] < x_tied[1:]]
print("mọi trung điểm:", all_midpoints.tolist())
print("ngưỡng tạo nhóm mới:", valid_midpoints.tolist())
assert np.allclose(valid_midpoints, [1.5, 3.0])

mọi trung điểm: [1.0, 1.5, 3.0]
ngưỡng tạo nhóm mới: [1.5, 3.0]


Ngưỡng 1.0 nằm giữa hai giá trị bằng nhau, nên không tách được hai hồ sơ có `x = 1`. Nó chia dữ liệu y hệt như ngưỡng 1.5. Chỉ có 1.5 và 3.0 là cho ra những cách chia khác nhau. Bước lọc này nhỏ, nhưng tự cài cây quyết định thì không được quên.

## Ví dụ 2: Giữ lại một tập để chọn độ sâu và mức cắt tỉa

Một cây mọc hết cỡ có thể học thuộc cả nhiễu. Ta tách `X_valid` ra trước khi fit. Sau đó so hai cách hãm cây: giới hạn độ sâu, và cắt tỉa bằng `ccp_alpha`. Các giá trị `ccp_alpha` lấy từ đường cắt tỉa mà scikit-learn tính sẵn trên train. Tập validation chỉ dùng để chọn cấu hình, không tham gia dựng cây.

In [ ]:
import numpy as np
from sklearn.datasets import make_classification
from sklearn.metrics import accuracy_score
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier

X, y = make_classification(
    n_samples=180, n_features=5, n_informative=3, n_redundant=0,
    class_sep=0.75, flip_y=0.12, random_state=14
)
X_train, X_valid, y_train, y_valid = train_test_split(
    X, y, test_size=0.30, stratify=y, random_state=9
)

depth_rows = []
for depth in [1, 2, 3, None]:
    tree = DecisionTreeClassifier(max_depth=depth, random_state=0).fit(X_train, y_train)
    depth_rows.append((depth, int(tree.get_n_leaves()),
                       accuracy_score(y_train, tree.predict(X_train)),
                       accuracy_score(y_valid, tree.predict(X_valid))))
print("depth / lá / train / valid:", [
    (d, leaves, round(train, 3), round(valid, 3))
    for d, leaves, train, valid in depth_rows
])

probe = DecisionTreeClassifier(random_state=0).fit(X_train, y_train)
path = probe.cost_complexity_pruning_path(X_train, y_train)
alphas = np.unique(path.ccp_alphas)
positions = np.linspace(0, len(alphas) - 1, num=min(7, len(alphas)), dtype=int)
alpha_candidates = alphas[positions]
pruned_rows = []
for alpha in alpha_candidates:
    tree = DecisionTreeClassifier(ccp_alpha=float(alpha), random_state=0).fit(
        X_train, y_train
    )
    pruned_rows.append((float(alpha), int(tree.get_n_leaves()),
                        accuracy_score(y_valid, tree.predict(X_valid))))
best_alpha, best_leaves, best_valid = max(pruned_rows, key=lambda row: row[2])
print("alpha / lá / valid:", [
    (round(alpha, 5), leaves, round(valid, 3))
    for alpha, leaves, valid in pruned_rows
])
print("alpha chọn:", round(best_alpha, 5),
      "lá:", best_leaves, "valid:", round(best_valid, 3))
assert 0.0 <= best_valid <= 1.0
assert best_leaves >= 1

depth / lá / train / valid: [(1, 2, 0.825, 0.685), (2, 4, 0.889, 0.704), (3, 7, 0.913, 0.704), (None, 25, 1.0, 0.648)]
alpha / lá / valid: [(0.0, 25, 0.648), (0.00661, 19, 0.685), (0.00762, 11, 0.722), (0.00927, 8, 0.704), (0.02116, 4, 0.704), (0.02549, 3, 0.685), (0.21177, 1, 0.5)]
alpha chọn: 0.00762 lá: 11 valid: 0.722


Đọc dòng đầu theo số lá. Cây không giới hạn có 25 lá và đúng hết train, nhưng validation chỉ còn 0.648, thấp nhất trong bốn cây. Điểm train tăng theo độ phức tạp là chuyện đương nhiên, nên chỉ cột validation mới giúp ta chọn.

Ở dòng thứ hai, `ccp_alpha` càng lớn thì cây càng bị cắt nhiều. Tại alpha = 0 cây vẫn đủ 25 lá. Alpha lớn nhất cắt cây còn đúng một lá, tức là đoán một nhãn cho mọi mẫu, và validation rơi về 0.5. Mức tốt nhất nằm ở giữa: 11 lá, validation 0.722. Nếu còn tiếp tục thử thêm nhiều cấu hình, cần giữ riêng một tập test để báo cáo kết quả cuối.

## Ví dụ 3: Feature importance không chứng minh nguyên nhân

Giả sử `gio_tu_hoc` là tín hiệu thật, còn `ma_ngau_nhien` là một mã số không mang thông tin gì. Một cây sâu vẫn có thể tìm ra vài ngưỡng tình cờ trên mã này. Ta so cây sâu với một cây bị giới hạn độ sâu và số mẫu tối thiểu ở mỗi lá. Sau đó tính permutation importance trên validation: xáo trộn từng cột và xem điểm giảm bao nhiêu.

In [ ]:
import numpy as np
from sklearn.inspection import permutation_importance
from sklearn.metrics import accuracy_score
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier

rng = np.random.default_rng(17)
n = 160
gio_tu_hoc = rng.uniform(0., 10., n)
ma_ngau_nhien = rng.permutation(n) / n
y = (gio_tu_hoc + rng.normal(0., 1.6, n) > 5.0).astype(int)
X = np.column_stack([gio_tu_hoc, ma_ngau_nhien])
X_train, X_valid, y_train, y_valid = train_test_split(
    X, y, test_size=0.30, stratify=y, random_state=6
)

deep = DecisionTreeClassifier(random_state=0).fit(X_train, y_train)
constrained = DecisionTreeClassifier(max_depth=3, min_samples_leaf=8,
                                     random_state=0).fit(X_train, y_train)
for name, model in [("sâu", deep), ("giới hạn", constrained)]:
    print(name, "lá:", model.get_n_leaves(),
          "train:", round(accuracy_score(y_train, model.predict(X_train)), 3),
          "valid:", round(accuracy_score(y_valid, model.predict(X_valid)), 3))

permutation = permutation_importance(
    deep, X_valid, y_valid, n_repeats=10, random_state=0,
    scoring="accuracy", n_jobs=1
)
print("impurity:", deep.feature_importances_.round(3).tolist())
print("permutation valid:", permutation.importances_mean.round(3).tolist())
assert permutation.importances_mean.shape == (2,)
assert np.isfinite(permutation.importances_mean).all()

sâu lá: 18 train: 1.0 valid: 0.729
giới hạn lá: 6 train: 0.902 valid: 0.771
impurity: [0.821, 0.179]
permutation valid: [0.229, -0.025]


Cây sâu đúng hết train nhưng thua cây giới hạn trên validation, 0.729 so với 0.771. Đó là dấu hiệu học thuộc.

Hai dòng importance kể hai câu chuyện khác nhau về mã ngẫu nhiên. `feature_importances_` tính trên train, và nó nói cây sâu đã dùng mã ngẫu nhiên cho khoảng 18% mức giảm Gini. Permutation importance tính trên validation thì gần 0, thậm chí hơi âm. Xáo trộn mã ngẫu nhiên không làm điểm giảm, nghĩa là những ngưỡng cây tìm được trên cột này không giúp gì cho dữ liệu mới. Dù vậy, cả hai con số vẫn chỉ mô tả model này trên dữ liệu này. Chúng không chứng minh giờ tự học là nguyên nhân của nhãn.

Một thí nghiệm nhỏ nên tự chạy: đổi `min_samples_leaf=8` thành `1` hoặc `16`, giữ nguyên cách chia train và validation. Nếu lá quá lớn làm validation giảm, ràng buộc đã xóa mất cả cấu trúc có ích. Nếu lá quá nhỏ làm train tăng mà validation giảm, cây lại quay về học thuộc.